# Notebook 01: Detection Fundamentals

**Module:** 08 Object Detection  
**Duration:** ~2 hrs

---

## Learning Objectives

1. Define object detection vs classification
2. Represent bounding boxes (xyxy, xywh, cxcywh)
3. Compute IoU between boxes
4. Connect to GeoSpatial pond/building counting

## Object Detection

**Classification:** What is in the image?

**Detection:** What objects are present AND where are they?

**Output:** Set of bounding boxes + class labels + confidence scores.

| Format | Fields | Used By |
|--------|--------|--------|
| xyxy | x_min, y_min, x_max, y_max | PyTorch, NMS |
| xywh | x_min, y_min, width, height | COCO annotations |
| cxcywh | center_x, center_y, w, h | YOLO, DETR |

**GeoSpatial:** Segmentation gives pixel masks; detection gives box counts for ponds, buildings, ships.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import torch
import torch.nn as nn
import torch.nn.functional as F

plt.rcParams['figure.figsize'] = (8, 5)
torch.manual_seed(42)
rng = np.random.default_rng(42)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

In [ ]:
def box_iou(boxes1, boxes2):
    """boxes: (N,4) and (M,4) in xyxy format. Returns (N,M)."""
    x1 = torch.max(boxes1[:, None, 0], boxes2[None, :, 0])
    y1 = torch.max(boxes1[:, None, 1], boxes2[None, :, 1])
    x2 = torch.min(boxes1[:, None, 2], boxes2[None, :, 2])
    y2 = torch.min(boxes1[:, None, 3], boxes2[None, :, 3])
    inter = (x2 - x1).clamp(min=0) * (y2 - y1).clamp(min=0)
    area1 = (boxes1[:, 2] - boxes1[:, 0]) * (boxes1[:, 3] - boxes1[:, 1])
    area2 = (boxes2[:, 2] - boxes2[:, 0]) * (boxes2[:, 3] - boxes2[:, 1])
    union = area1[:, None] + area2[None, :] - inter
    return inter / (union + 1e-7)


def xywh_to_xyxy(boxes):
    x, y, w, h = boxes.unbind(-1)
    return torch.stack([x, y, x + w, y + h], dim=-1)


def xyxy_to_cxcywh(boxes):
    x1, y1, x2, y2 = boxes.unbind(-1)
    return torch.stack([(x1 + x2) / 2, (y1 + y2) / 2, x2 - x1, y2 - y1], dim=-1)

In [ ]:
# Visualize boxes and IoU
fig, ax = plt.subplots(figsize=(6, 6))
ax.set_xlim(0, 100); ax.set_ylim(0, 100); ax.invert_yaxis()
box_a = torch.tensor([20., 20., 60., 60.])
box_b = torch.tensor([40., 40., 80., 80.])
for b, c in [(box_a, 'tab:blue'), (box_b, 'tab:orange')]:
    x1, y1, x2, y2 = b.tolist()
    ax.add_patch(patches.Rectangle((x1, y1), x2-x1, y2-y1, fill=False, edgecolor=c, lw=2))
iou = box_iou(box_a.unsqueeze(0), box_b.unsqueeze(0)).item()
ax.set_title(f'IoU = {iou:.3f}'); plt.show()

## IoU (Intersection over Union)

$$\text{IoU} = \frac{|A \cap B|}{|A \cup B|}$$

Used for matching predictions to ground truth and for NMS.

---

## Summary

Detection = classify + localize. IoU measures box overlap.

**Next:** [02_NMS_and_mAP.ipynb](02_NMS_and_mAP.ipynb)